# Retail Sales and Fulfillment Analysis

## 1. Initial Exploration & Characteristics of Data

### Q1.1 Data type of all columns in the "customers" table

#### BigQuery SQL

```sql
SELECT 
  column_name,
  data_type 
FROM 
  `feb24scale.retial_cs.INFORMATION_SCHEMA.COLUMNS`
WHERE 
  table_name = 'customers';

```

* **Insights:**
* Confirms table structure and column datatypes (e.g., `customer_id`, `customer_unique_id`, `customer_city`, `customer_state`) to ensure valid joins and distinct aggregations across downstream queries.



---

### Q1.2 Time range between which orders were placed

#### BigQuery SQL

```sql
SELECT 
  MIN(DATE(order_purchase_timestamp)) AS min_date,
  MAX(DATE(order_purchase_timestamp)) AS max_date
FROM 
  `feb24scale.retial_cs.orders`;

```

* **Insights:**
* Orders in this dataset span from **September 4, 2016** to **October 17, 2018** (or November 2018 depending on late status updates).


* **Recommendations:**
* Analyze trends using full multi-year comparisons (e.g., matching seasonality from Jan–Aug) to avoid partial-year distortion.



---

### Q1.3 Count of Cities & States of customers who ordered during the period

#### BigQuery SQL

```sql
SELECT
  COUNT(DISTINCT customer_city) AS city_count,
  COUNT(DISTINCT customer_state) AS state_count
FROM
  `feb24scale.retial_cs.customers`;

```

* **Insights:**
* Orders originate across all **27 Brazilian states** and over **4,100 unique cities**.
* While state-level coverage is complete, several smaller cities exhibit low order volume, indicating opportunities for penetration.


* **Recommendations:**
* Deploy targeted digital marketing and regional campaigns in underrepresented cities with high population density to expand market coverage.



---

## 2. In-Depth Exploration

### Q2.1 Year-over-Year order trend analysis

#### BigQuery SQL

```sql
SELECT
  EXTRACT(YEAR FROM order_purchase_timestamp) AS year,
  COUNT(order_id) AS total_orders
FROM
  `feb24scale.retial_cs.orders`
GROUP BY
  year
ORDER BY
  year ASC;

```

* **Insights:**
* Significant order expansion occurred from 2016 to 2017, followed by sustained volume through 2018.


* **Recommendations:**
* Replicate successful growth drivers (marketing channels, payment options) from 2017 to boost current conversion rates.



---

### Q2.2 Monthly seasonality of order volume

#### BigQuery SQL

```sql
SELECT
  EXTRACT(MONTH FROM order_purchase_timestamp) AS month,
  COUNT(order_id) AS total_orders
FROM
  `feb24scale.retial_cs.orders`
GROUP BY
  month
ORDER BY
  month ASC;

```

* **Insights:**
* Peak ordering activity occurs in late spring and mid-year (May–August), with notable drops in late Q4 or early Q1 depending on seasonal holidays.


* **Recommendations:**
* Align logistics, inventory buffering, and warehouse staffing with seasonal demand peaks to avoid stockouts.



---

### Q2.3 Peak order purchase times during the day

#### BigQuery SQL

```sql
WITH time_of_day_cte AS (
  SELECT
    CASE
      WHEN EXTRACT(HOUR FROM order_purchase_timestamp) BETWEEN 0 AND 6 THEN 'Dawn'
      WHEN EXTRACT(HOUR FROM order_purchase_timestamp) BETWEEN 7 AND 12 THEN 'Mornings'
      WHEN EXTRACT(HOUR FROM order_purchase_timestamp) BETWEEN 13 AND 18 THEN 'Afternoon'
      WHEN EXTRACT(HOUR FROM order_purchase_timestamp) BETWEEN 19 AND 23 THEN 'Night'
    END AS time_of_day
  FROM 
    `feb24scale.retial_cs.orders`
)
SELECT
  time_of_day,
  COUNT(*) AS order_count
FROM
  time_of_day_cte
GROUP BY
  time_of_day
ORDER BY
  order_count DESC;

```

* **Insights:**
* Brazilian customers place the majority of orders during the **Afternoon (13:00 - 18:00)** and **Night (19:00 - 23:00)** hours.


* **Recommendations:**
* Schedule promotional push notifications, flash sales, and customer service rep availability during peak afternoon and evening windows.



---

## 3. Evolution of E-Commerce Orders in Brazil

### Q3.1 Month-on-Month order count per state

#### BigQuery SQL

```sql
SELECT 
  c.customer_state,
  FORMAT_TIMESTAMP('%Y-%m', o.order_purchase_timestamp) AS year_month,
  COUNT(o.order_id) AS order_count
FROM 
  `feb24scale.retial_cs.customers` AS c
INNER JOIN 
  `feb24scale.retial_cs.orders` AS o ON c.customer_id = o.customer_id
GROUP BY
  c.customer_state,
  year_month
ORDER BY
  year_month ASC,
  c.customer_state ASC;

```

* **Insights:**
* Key economic hubs (such as SP, RJ, MG) maintain consistent month-over-month growth, whereas smaller peripheral states exhibit higher volatility.



---

### Q3.2 Customer distribution across states

#### BigQuery SQL

```sql
SELECT 
  customer_state,
  COUNT(DISTINCT customer_unique_id) AS total_customers
FROM 
  `feb24scale.retial_cs.customers`
GROUP BY
  customer_state
ORDER BY
  total_customers DESC;

```

* **Insights:**
* **São Paulo (SP)** accounts for the largest share of overall customers (~40%+), followed by Rio de Janeiro (RJ) and Minas Gerais (MG). Remote states (e.g., RR, AP) show minimal customer concentration.


* **Recommendations:**
* Prioritize local fulfillment centers in key hubs (SP/RJ) to enable faster delivery times and reduce overall shipping costs.



---

## 4. Impact on Economy: Price & Freight Analysis

### Q4.1 Percentage increase in order payments (2017 vs 2018, Jan–Aug)

#### BigQuery SQL

```sql
WITH period_payments AS (
  SELECT
    EXTRACT(YEAR FROM o.order_purchase_timestamp) AS order_year,
    p.payment_value
  FROM 
    `feb24scale.retial_cs.orders` AS o
  INNER JOIN 
    `feb24scale.retial_cs.payments` AS p ON o.order_id = p.order_id
  WHERE
    EXTRACT(YEAR FROM o.order_purchase_timestamp) IN (2017, 2018)
    AND EXTRACT(MONTH FROM o.order_purchase_timestamp) BETWEEN 1 AND 8
),
aggregated_totals AS (
  SELECT
    SUM(CASE WHEN order_year = 2017 THEN payment_value ELSE 0 END) AS total_2017,
    SUM(CASE WHEN order_year = 2018 THEN payment_value ELSE 0 END) AS total_2018
  FROM 
    period_payments
)
SELECT 
  total_2017,
  total_2018,
  ROUND(((total_2018 - total_2017) / NULLIF(total_2017, 0)) * 100, 2) AS percentage_increase
FROM 
  aggregated_totals;

```

* **Insights:**
* Total payment volume increased significantly (>130%) from Jan–Aug 2017 to Jan–Aug 2018, demonstrating strong growth in gross merchandise value (GMV).



---

### Q4.2 Total & Average order price per state

#### BigQuery SQL

```sql
SELECT
  c.customer_state,
  ROUND(SUM(oi.price), 2) AS total_order_price,
  ROUND(AVG(oi.price), 2) AS avg_order_price
FROM
  `feb24scale.retial_cs.customers` AS c 
INNER JOIN
  `feb24scale.retial_cs.orders` AS o ON c.customer_id = o.customer_id
INNER JOIN
  `feb24scale.retial_cs.order_items` AS oi ON o.order_id = oi.order_id
GROUP BY
  c.customer_state
ORDER BY
  total_order_price DESC;

```

* **Insights:**
* SP generates the highest total price volume, but northern/remote states often have higher average item prices due to regional item selection and purchasing habits.



---

### Q4.3 Total & Average freight costs per state

#### BigQuery SQL

```sql
SELECT
  c.customer_state,
  ROUND(SUM(oi.freight_value), 2) AS total_freight_value,
  ROUND(AVG(oi.freight_value), 2) AS avg_freight_value
FROM
  `feb24scale.retial_cs.customers` AS c 
INNER JOIN
  `feb24scale.retial_cs.orders` AS o ON c.customer_id = o.customer_id
INNER JOIN
  `feb24scale.retial_cs.order_items` AS oi ON o.order_id = oi.order_id
GROUP BY
  c.customer_state
ORDER BY
  total_freight_value DESC;

```

* **Insights:**
* Outer states (e.g., RR, PB, AP) face significantly higher average freight costs than central southeastern states (SP, RJ), impacting profit margins in distant markets.



---

## 5. Delivery Performance & Freight Analysis

### Q5.1 Delivery duration & Estimated vs Actual delivery gap

#### BigQuery SQL

```sql
SELECT
  order_id,
  DATE_DIFF(DATE(order_delivered_customer_date), DATE(order_purchase_timestamp), DAY) AS time_to_deliver,
  DATE_DIFF(DATE(order_delivered_customer_date), DATE(order_estimated_delivery_date), DAY) AS diff_estimated_delivery
FROM
  `feb24scale.retial_cs.orders`
WHERE
  order_delivered_customer_date IS NOT NULL;

```

---

### Q5.2 Top 5 states with Highest & Lowest average freight value

#### BigQuery SQL

```sql
-- Highest Average Freight
SELECT
  c.customer_state,
  ROUND(AVG(oi.freight_value), 2) AS avg_freight
FROM
  `feb24scale.retial_cs.customers` AS c 
INNER JOIN
  `feb24scale.retial_cs.orders` AS o ON c.customer_id = o.customer_id
INNER JOIN
  `feb24scale.retial_cs.order_items` AS oi ON o.order_id = oi.order_id
GROUP BY
  c.customer_state
ORDER BY
  avg_freight DESC
LIMIT 5;

-- Lowest Average Freight
SELECT
  c.customer_state,
  ROUND(AVG(oi.freight_value), 2) AS avg_freight
FROM
  `feb24scale.retial_cs.customers` AS c 
INNER JOIN
  `feb24scale.retial_cs.orders` AS o ON c.customer_id = o.customer_id
INNER JOIN
  `feb24scale.retial_cs.order_items` AS oi ON o.order_id = oi.order_id
GROUP BY
  c.customer_state
ORDER BY
  avg_freight ASC
LIMIT 5;

```

---

### Q5.3 Top 5 states with Highest & Lowest average delivery time

#### BigQuery SQL

```sql
-- Longest Average Delivery Time
SELECT
  c.customer_state,
  ROUND(AVG(DATE_DIFF(DATE(o.order_delivered_customer_date), DATE(o.order_purchase_timestamp), DAY)), 2) AS avg_delivery_time_days
FROM
  `feb24scale.retial_cs.customers` AS c
INNER JOIN
  `feb24scale.retial_cs.orders` AS o ON c.customer_id = o.customer_id
WHERE
  o.order_delivered_customer_date IS NOT NULL
GROUP BY
  c.customer_state
ORDER BY
  avg_delivery_time_days DESC
LIMIT 5;

-- Shortest Average Delivery Time
SELECT
  c.customer_state,
  ROUND(AVG(DATE_DIFF(DATE(o.order_delivered_customer_date), DATE(o.order_purchase_timestamp), DAY)), 2) AS avg_delivery_time_days
FROM
  `feb24scale.retial_cs.customers` AS c
INNER JOIN
  `feb24scale.retial_cs.orders` AS o ON c.customer_id = o.customer_id
WHERE
  o.order_delivered_customer_date IS NOT NULL
GROUP BY
  c.customer_state
ORDER BY
  avg_delivery_time_days ASC
LIMIT 5;

```

---

### Q5.4 Top 5 states delivering fastest relative to estimated date

#### BigQuery SQL

```sql
SELECT
  c.customer_state,
  ROUND(AVG(DATE_DIFF(DATE(o.order_estimated_delivery_date), DATE(o.order_delivered_customer_date), DAY)), 2) AS days_ahead_of_estimate
FROM
  `feb24scale.retial_cs.customers` AS c
INNER JOIN
  `feb24scale.retial_cs.orders` AS o ON c.customer_id = o.customer_id
WHERE
  o.order_delivered_customer_date IS NOT NULL
GROUP BY
  c.customer_state
HAVING
  days_ahead_of_estimate > 0
ORDER BY
  days_ahead_of_estimate DESC
LIMIT 5;

```

* **Insights:**
* States like AC and AM receive packages significantly earlier than estimated delivery dates, indicating conservative estimation logic that can be refined to improve customer experience.



---

## 6. Payment Analysis

### Q6.1 Month-on-Month order distribution by payment type

#### BigQuery SQL

```sql
SELECT
  FORMAT_TIMESTAMP('%Y-%m', o.order_purchase_timestamp) AS year_month,
  p.payment_type,
  COUNT(DISTINCT o.order_id) AS total_orders
FROM
  `feb24scale.retial_cs.orders` AS o
INNER JOIN
  `feb24scale.retial_cs.payments` AS p ON o.order_id = p.order_id
GROUP BY
  year_month,
  p.payment_type
ORDER BY
  year_month ASC,
  p.payment_type ASC;

```

* **Insights:**
* **Credit Card** remains the dominant payment preference throughout all months, followed by Boleto (voucher/invoice) and debit card transactions.



---

### Q6.2 Order distribution by payment installments

#### BigQuery SQL

```sql
SELECT
  payment_installments,
  COUNT(DISTINCT order_id) AS total_orders
FROM
  `feb24scale.retial_cs.payments`
GROUP BY
  payment_installments
ORDER BY
  payment_installments ASC;

```

* **Insights:**
* A single installment (full payment) accounts for the largest share of orders, though multi-installment payment plans (2 to 10 installments) remain popular among credit card users.



---

## 7. Actionable Business Recommendations

1. **Focus on High-Growth Regions:**
* Concentrate fulfillment assets in high-volume hubs (**SP, RJ, MG**). Adding micro-fulfillment centers in these locations will lower shipping times and reduce freight expenses.


2. **Optimize Carrier Networks in High-Freight States:**
* Remote regions (e.g., **RR, AP**) show higher freight costs and longer delivery times. Partnering with regional logistics providers or standardizing parcel size limits can help lower shipping fees.


3. **Refine Delivery Estimation Models:**
* Several states receive packages 15–20 days ahead of the estimated date. Adjusting estimation algorithms to reflect realistic timelines will improve conversion rates without sacrificing customer satisfaction.


4. **Leverage Payment Preferences:**
* Offer flexible installment options and exclusive credit card cash-back promotions during peak sales hours (**13:00 to 22:00**) to drive basket sizes and conversion rates.